# Track 05 — Memory & Long Context

**구성** : 각 `Session`은 코드 설명(텍스트) -> 코드 -> 해석(텍스트) 순으로 정리되어 있습니다.

**목표** : Track 05의 목표는 컨텍스트 예산·압축과 Memory Ledger·세션 스냅샷으로 긴 대화를 다루는 것입니다.

**산출물:** `_out/compaction_report.json`, `_out/session_state.json`


In [ ]:
import json
import os
import sys
from pathlib import Path

import logging
import warnings
# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽게 라이브러리 로그를 줄인다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e .
# (kr) 저장소 루트에서 editable 설치 필요: pip install -r requirements.txt && pip install -e .
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone 가 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e . 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
import exaone.context_management.constants

TRACK05 = ROOT / "recipes" / "track05_memory_and_long_context"
DATA = TRACK05 / "data"
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)
print("exaone", exaone.__version__)
print("DATA:", DATA)

exaone 0.1.0
DATA: <cookbook-root>/recipes/track05_memory_and_long_context/data


**출력 해석:** `exaone` 버전과 경로(`DATA`)가 보이면 라이브러리·경로가 준비된 것입니다. 이 노트북은 LLM 을 호출하지 않으므로(압축 단계는 mock) 모델·API 키는 필요 없습니다.

## Session 1. 컨텍스트 예산 & 압축

LLM 한 번 호출의 토큰 예산은 입력 토큰 + 예약한 출력 토큰(reserved_new_tokens) 입니다. 둘의 합이 모델 컨텍스트 한도를 넘으면 안 됩니다.


### Session 1-1. 컨텍스트 상수

**하는 일:** 컨텍스트 예산을 결정하는 네 가지 상수를 출력해, 이 노트북이 어떤 한도 안에서 도는지 먼저 확인합니다.

**정상:** `MAX tokens:`, `RECOMMENDED tokens:`, `TOOL verbatim cap:`, `chars/token estimate:` 네 값이 출력됩니다.

**의미:** 이 값들은 `.env` 의 `CORE_CONTEXT_LENGTH_*` 등에서 와서 `exaone.context_management.constants` 에 반영됩니다. 이후 압축·컷오프(1-2~1-7)가 전부 이 한도를 기준으로 동작하므로, 무엇이 한도인지부터 봅니다.

In [2]:
print("MAX tokens:", exaone.context_management.CONTEXT_LENGTH_MAX_TOKENS)
print("RECOMMENDED tokens:", exaone.context_management.CONTEXT_LENGTH_RECOMMENDED_TOKENS)
print("TOOL verbatim cap:", exaone.context_management.constants.CONTEXT_TOOL_VERBATIM_MAX_TOKENS)
print("chars/token estimate:", exaone.context_management.CHARS_PER_TOKEN_ESTIMATE)


MAX tokens: 256000
RECOMMENDED tokens: 128000
TOOL verbatim cap: 4096
chars/token estimate: 4


**출력 해석:** 네 상수가 컨텍스트 예산을 정합니다.

- **MAX tokens (256000)** — 모델이 한 번 호출에서 받을 수 있는 입력+출력 토큰의 **절대 상한**. 넘으면 호출 자체가 실패합니다. (`CORE_CONTEXT_LENGTH_MAX_TOKENS`)
- **RECOMMENDED tokens (128000)** — 안정적으로 쓰도록 권장하는 **운영 한도**. MAX 보다 낮게 둬 품질·지연·비용에 여유를 줍니다. 압축 목표선으로 씁니다. (`CORE_CONTEXT_LENGTH_RECOMMENDED_TOKENS`)
- **TOOL verbatim cap (4096)** — 도구 결과 **하나**를 컨텍스트에 원문 그대로 넣을 수 있는 최대 토큰. 넘으면 통째로 넣지 않고 artifact 로 뺍니다(1-7·Session 2). (`CORE_CONTEXT_TOOL_VERBATIM_MAX_TOKENS`)
- **chars/token estimate (4)** — 토크나이저를 돌리지 않고 *문자 수 ÷ 4* 로 토큰 수를 빠르게 어림하는 계수. 압축 판단을 가볍게 하기 위한 근사값입니다. (`CORE_CHARS_PER_TOKEN_ESTIMATE`)

요약: MAX = 못 넘는 벽, RECOMMENDED = 평소 목표선, verbatim cap = 도구 결과 한 건 한도, chars/token = 토큰 수 어림 계수.

**설정:** 위 값은 `exaone/config.py` 의 **기본값**이고, 루트 `.env` 에서 해당 `CORE_...` 키를 적으면 해당 값이 적용됩니다.

### Session 1-2. 입력 + 출력 예산 계산

**하는 일:** 모델을 호출하기 전, 입력 및 출력 토큰 설정값의 합이 `MAX_tokens` 를 초과하지 않도록 제어하는 헬퍼 함수의 작동 방식을 3가지 케이스별로 검증합니다.

**정상:** 세 케이스의 `capped`·`max_input_slot`·`validate` 결과가 출력됩니다.

**의미:** 한 호출 예산 = **입력 토큰 + 예약 출력 토큰**, 합이 MAX 를 넘으면 안 됩니다. 출력 예약은 `cap_max_new_tokens` 가 **남은 공간에 맞춰 자동으로 줄여** 넘깁니다. 단 입력 자체가 넘치면 여기선 `validate_input_tokens` 가 **에러로 알릴 뿐**, 입력을 실제로 줄이는 건 1-5(hard_cap)·1-6(압축)입니다.

In [3]:
MAX = exaone.context_management.CONTEXT_LENGTH_MAX_TOKENS


def show_budget(label, input_tokens, requested_new=4096):
    # (en) cap output reservation -> compute input slot -> validate input + reservation
    # (kr) 출력 예약 캡 -> 입력 상한 계산 -> 입력+예약 검증
    capped = exaone.context_management.cap_max_new_tokens(input_tokens, requested_new)
    slot = exaone.context_management.max_input_tokens_for_context(
        max_context_tokens=MAX, reserved_new_tokens=capped,
    )
    err = exaone.context_management.validate_input_tokens(input_tokens, reserved_new_tokens=capped)
    print(f"[{label}] input={input_tokens} | req_new={requested_new} -> capped={capped} | max_input_slot={slot}")
    print(f"          validate: {err or 'OK (None)'}")


print("MAX tokens:", MAX, "\n")
show_budget("정상", 120000)                # (kr) 여유 충분 — 출력 예약 그대로, 통과
show_budget("출력 예약 캡", MAX - 2000)     # (kr) 입력이 커서 남은 2000 으로 출력이 깎임
show_budget("입력 한도 초과", MAX + 4000)   # (kr) 입력 자체가 한도 초과 — validate 가 에러로 탐지


MAX tokens: 256000 

[정상] input=120000 | req_new=4096 -> capped=4096 | max_input_slot=251904
          validate: OK (None)
[출력 예약 캡] input=254000 | req_new=4096 -> capped=2000 | max_input_slot=254000
          validate: OK (None)
[입력 한도 초과] input=260000 | req_new=4096 -> capped=256 | max_input_slot=255744
          validate: 입력 컨텍스트가 너무 깁니다 (예상 입력 260000 + 생성 예약 256 = 260256 토큰). 최대 256000 토큰 이하로 줄여 주세요. 긴 대화는 요약하거나 오래된 턴을 제거한 뒤 다시 시도해 주세요.


**출력 해석:** 입력 크기에 따라 동작이 셋으로 갈립니다. (출력 예약은 `4096` 고정)

- **정상 (input 120000)** — 여유가 커서 출력 예약 `4096` 그대로, `validate: None`(통과).
- **출력 예약 캡 (input 254000)** — 남은 공간이 2000 뿐이라 `cap_max_new_tokens` 가 출력을 `4096 → 2000` 으로 **깎아서** 넘깁니다. 입력+예약=256000 으로 딱 한도라 `validate: None`.
- **입력 한도 초과 (input 260000)** — 입력만으로 이미 한도 초과. 출력을 최소(256)로 깎아도 `validate` 가 **에러 메시지**를 반환("…줄여 주세요").

핵심: **출력 예약은 자동으로 깎여 넘어가지만(cap), 입력 초과는 여기선 "탐지"만** 합니다. 실제로 입력을 줄이는 건 1-5 `hard_cap_messages`(강제 컷)·1-6 압축입니다.

### Session 1-3. transcript 로드 + 턴별 토큰 누적

**하는 일:** 회의록(`meeting_transcript.json`)에 포함된 `user`와 `assistant`의 `content` 를 순서대로 `messages` 로 쌓으면서, 턴마다 누적 토큰(`estimate_tokens_from_messages`)을 재 `token_curve` 를 만듭니다.

**입력:** `data/meeting_transcript.json` (12 segments)

**정상:** `시나리오:`·`segments:`·`title:`·`turns:`·`final tokens:` 가 출력됩니다.

**의미:** API 없이도 멀티턴이 쌓이며 토큰이 어떻게 누적되는지 보기 위한 고정 회의록입니다. 이 `token_curve` 를 1-4(그래프)·1-5~6(압축)이 이어 씁니다.

In [4]:
transcript = json.loads((DATA / "meeting_transcript.json").read_text(encoding="utf-8"))
print("시나리오: 긴 회의록", transcript.get("title", "meeting_transcript.json"))
print("  segments:", len(transcript.get("segments", [])))
messages = [{"role": "system", "content": "You are a meeting note assistant. Summarize decisions and action items."}]
token_curve = []
for seg in transcript["segments"]:
    messages.append({"role": seg["role"], "content": seg["content"]})
    tokens = exaone.context_management.estimate_tokens_from_messages(messages)
    token_curve.append({"turn": len(token_curve) + 1, "role": seg["role"], "tokens_total": tokens})
print("title:", transcript["title"])
print("turns:", len(token_curve), "| final tokens:", token_curve[-1]["tokens_total"])


시나리오: 긴 회의록 2026 Q1 제품 기획 회의
  segments: 12


title: 2026 Q1 제품 기획 회의
turns: 12 | final tokens: 573


**출력 해석:** 회의록 12턴을 system 프롬프트 부터 차례로 쌓으며 누적 토큰을 잰 결과입니다(`final tokens: 573`). 여기 573 은 데모용 짧은 회의록이라 작지만, 실제로는 이 값이 MAX 에 가까워질 때 압축이 필요해집니다.

### Session 1-4. 토큰 사용 추이 (ASCII)

**하는 일:** 1-3 에서 만든 `token_curve` 를 ASCII 막대로 그려, 턴이 쌓일수록 누적 토큰이 어떻게 느는지 한눈에 봅니다.

**정상:** 12줄(`turn · role · tokens_total · 막대`)이 출력됩니다.

**의미:** 멀티턴은 매 턴 지난 대화 전체를 다시 입력으로 넣으므로 누적 토큰이 **단조 증가**합니다(1-3 곡선). 실제 긴 대화라면 이 곡선이 RECOMMENDED·MAX 에 닿는 지점에서 압축(1-5·1-6)이 필요해집니다.

In [5]:
def ascii_bar(value, scale=40, ref=None):
    ref = ref or max(item["tokens_total"] for item in token_curve)
    width = max(1, int(value / max(ref, 1) * scale))
    return "█" * width


ref_tokens = token_curve[-1]["tokens_total"]
for row in token_curve:
    print(f"turn {row['turn']:>2} {row['role']:<10} {row['tokens_total']:>5}  {ascii_bar(row['tokens_total'], ref=ref_tokens)}")


turn  1 user          69  ████
turn  2 assistant    123  ████████
turn  3 user         169  ███████████
turn  4 user         218  ███████████████
turn  5 assistant    261  ██████████████████
turn  6 user         308  █████████████████████
turn  7 user         359  █████████████████████████
turn  8 assistant    386  ██████████████████████████
turn  9 user         435  ██████████████████████████████
turn 10 user         491  ██████████████████████████████████
turn 11 assistant    536  █████████████████████████████████████
turn 12 user         573  ████████████████████████████████████████


**출력 해석:** 턴마다 막대가 길어지면 누적 입력이 계속 커지는 것입니다(여기 12턴 → 573 토큰). 이 추이가 한도에 가까워질수록 다음 셀들의 압축·컷이 개입합니다.

### Session 1-5. 호출 직전 정리 — `prepare_messages_for_llm_chat`

**하는 일:** 실제 LLM 호출 직전에 메시지·출력 예약을 한도(MAX)에 맞추는 `prepare_messages_for_llm_chat` 를, 입력/출력 크기를 바꿔 **통과 / 출력 예약 조정 / 입력 하드캡** 세 케이스로 봅니다.

**정상:** 세 케이스의 `msgs … -> …`·`reserved … -> …` 변화가 출력됩니다.

**의미:** 이 함수 하나가 **자동으로** 두 단계로 맞춥니다 — ① 먼저 **출력 예약을 줄여** 입력+예약을 MAX 안에 넣습니다. ② 출력 예약을 최소로 줄여도 **입력 자체가 안 들어갈 때(입력이 사실상 MAX 이상)** 그제서야 오래된 메시지부터 버려(hard cap) 맞춥니다. (system 은 보존).

In [6]:
MAX = exaone.context_management.CONTEXT_LENGTH_MAX_TOKENS


def show_prepare(label, msgs, reserved_new):
    # (en) one call caps output reservation AND hard-caps input if over MAX
    # (kr) 이 한 번 호출이 출력 예약 캡 + (초과 시) 입력 hard cap 까지 한다
    before = exaone.context_management.estimate_tokens_from_messages(msgs)
    prepared, capped = exaone.context_management.prepare_messages_for_llm_chat(
        msgs, reserved_new_tokens=reserved_new, max_context_tokens=MAX,
    )
    after = exaone.context_management.estimate_tokens_from_messages(prepared)
    print(f"[{label}] msgs {before}->{after} tok ({len(msgs)}->{len(prepared)}개) | reserved {reserved_new}->{capped}")


# (kr) 1) 여유 충분 -> 그대로 통과
show_prepare("통과", messages, 4096)
# (kr) 2) 출력 예약이 과도 -> 남은 공간에 맞춰 reserved 자동 캡
show_prepare("출력 예약 조정", messages, 300_000)
# (kr) 3) 입력이 MAX 초과(거대 메시지 삽입) -> 오래된 메시지부터 버려 입력을 맞춤
oversized = [messages[0], {"role": "user", "content": "공지 " * 90_000}] + messages[1:]
show_prepare("입력 하드캡", oversized, 4096)


[통과] msgs 573->573 tok (13->13개) | reserved 4096->4096
[출력 예약 조정] msgs 573->573 tok (13->13개) | reserved 300000->255427


[입력 하드캡] msgs 270576->573 tok (14->13개) | reserved 4096->4096


**출력 해석:** 같은 함수가 입력에 따라 다르게 동작합니다.

- **통과 (reserved 4096)** — 573 토큰은 MAX(256000)에 한참 못 미쳐 **무변화**(메시지·예약 그대로).
- **출력 예약 조정 (reserved 300000)** — 출력 예약이 과도하면 남은 공간에 맞춰 `300000 → 255427` 로 **자동 캡**. 입력은 그대로.
- **입력 하드캡 (거대 메시지 삽입)** — 거대 메세지를 하나 첫 부분에 추가하여 입력이 270576 으로 MAX 초과 → `prepare` 가 **오래된 거대 메시지를 버려** 입력을 573 으로 맞춤(14→13개). 입력을 줄여 공간이 생겼으므로 출력 예약은 요청값 `4096` 그대로 유지됩니다.

핵심: 실사용에선 **`prepare_messages_for_llm_chat` 한 번만 부르면** 출력 캡·입력 하드캡이 다 적용되고, system 은 어느 경우든 보존됩니다. (`input + reserved ≤ MAX` 항상 유지)

### Session 1-6. LLM 압축 — `compress_messages_for_turn`

**하는 일:** 전체 입력 중 오래된 대화를 **요약**해 메시지 수·토큰을 줄이는 `compress_messages_for_turn` 을 봅니다. 데모로 LLM 없이 `MOCK_SUMMARY`를 대화 요약으로 사용합니다. (실사용 시 `patch`와 `MOCK_SUMMARY`를 제거하고 `llm=` 에 입력할 모델 필요)

**정상:** `compressed tokens:` 와 `message count: before N -> after M` 이 출력됩니다.

**의미:** `keep_last_n=2` 라 **최근 2개 턴은 그대로** 두고, 그보다 오래된 대화는 `[이전 대화 요약]` 한 덩이(system)로 합칩니다(요약이라 세부는 버리는 *lossy* 압축). 

In [7]:
from unittest.mock import patch

MOCK_SUMMARY = "Q1 로드맵: CDN/FAQ/SSO 우선. 예산 4.2억 초안 유지. 액션 — Eng CDN POC, Docs FAQ 번역, Finance breakdown, SSO 보안 HITL."

# (en) Patch the internal LLM compressor so the demo is deterministic and offline.
# (kr) 내부 LLM 압축기를 patch 해 데모가 결정적이고 오프라인으로 동작하게 한다.
# (kr) patch 는 compress_messages_for_turn 이 내부에서 부르는 _compress_with_llm 을 가짜로
#      바꿔치기해서, 인자 전달 없이도 그 내부 호출 결과가 MOCK_SUMMARY 가 되게 만든다.
with patch("exaone.context_management.messages._compress_with_llm", return_value=(MOCK_SUMMARY, True)):
    compressed = exaone.context_management.compress_messages_for_turn(
        messages,
        llm=None,
        keep_last_n=2,
        target_max_tokens=120,
        reserved_new_tokens=256,
    )
compressed_tokens = exaone.context_management.estimate_tokens_from_messages(compressed)
print("compressed tokens:", compressed_tokens)
print("message count: before", len(messages), "-> after", len(compressed))
for msg in compressed[:3]:
    role = msg.get("role") if isinstance(msg, dict) else getattr(msg, "role", "?")
    content = msg.get("content") if isinstance(msg, dict) else getattr(msg, "content", "")
    print(f"  [{role}] {str(content)[:100].replace(chr(10), ' ')}")


compressed tokens: 169
message count: before 13 -> after 4
  [system] You are a meeting note assistant. Summarize decisions and action items.
  [system] [이전 대화 요약] Q1 로드맵: CDN/FAQ/SSO 우선. 예산 4.2억 초안 유지. 액션 — Eng CDN POC, Docs FAQ 번역, Finance breakdown, 
  [assistant] 리스크: SSO 일정 — 보안 승인 지연 시 4월로 slip. Escalation 경로는 CISO 에게 사전 공유.


**출력 해석:** 13개 메시지가 **4개**로, 573 → **169 토큰**으로 줄었습니다. 구성은 `system`(원래 지시) + `[이전 대화 요약]`(오래된 턴을 합친 요약) + 최근 2개 턴. 오래된 맥락을 요약으로 대체해 메시지 수·토큰을 줄입니다.  
참고로 **1-7 과의 차이:** 1-6 은 **누적된 대화 전체**를 요약으로 줄이고, 1-7 은 **개별 도구 결과 하나**가 한도를 넘는지 판정해 artifact 로 빼는 것 — 대상·방식이 다릅니다.

### Session 1-7. 도구 결과 verbatim cap (개념)

**하는 일:** 도구 결과 하나의 토큰이 `CONTEXT_TOOL_VERBATIM_MAX_TOKENS` 를 넘는지 **판정**합니다(자르지 않고 넘는지 여부만).

**정상:** `single tool run tokens:`·`CONTEXT_TOOL_VERBATIM_MAX_TOKENS:`·`needs verbatim cap (-> artifact): True/False` 가 출력됩니다.

**의미:** 도구 하나가 거대한 결과를 통째로 컨텍스트에 넣으면 토큰 수가 `MAX_TOKENS` 를 한 번에 초과 할 수 있습니다. 미리 정해진 한도를 넘는 도구 결과는 본문을 직접 넣지 말고 **Session 2 의 artifact 로 빼고 ledger 엔 참조만** 남겨야 합니다. 

In [8]:
big_tool_body = "row " * 8000
tool_run = [
{"role": "assistant", "content": "calling analytics_export", "tool_calls": [{"id": "1", "type": "function"}]},
{"role": "tool", "content": big_tool_body},
]
tool_tokens = exaone.context_management.estimate_tokens_from_messages(tool_run)
cap = exaone.context_management.constants.CONTEXT_TOOL_VERBATIM_MAX_TOKENS
print("single tool run tokens:", tool_tokens)
print("CONTEXT_TOOL_VERBATIM_MAX_TOKENS:", cap)
print("needs verbatim cap (-> artifact):", tool_tokens > cap)


single tool run tokens: 8009
CONTEXT_TOOL_VERBATIM_MAX_TOKENS: 4096
needs verbatim cap (-> artifact): True


**출력 해석:** 도구 결과가 8009 토큰으로 cap(4096)을 넘어 `needs verbatim cap: True` — 이 결과는 컨텍스트에 원문으로 넣으면 안 되고 artifact 로 빼야 한다는 신호입니다. 여기선 판정만 하고, 실제 처리는 Session 2 `store_large_tool_result` 가 합니다.

### Session 1-8. 산출물 — `compaction_report.json`

**하는 일:** Session 1 결과(상수·`token_curve`·압축 전후 토큰)를 `compaction_report.json` 으로 저장합니다.

**정상:** 저장 경로가 `saved:` 로 출력됩니다.

**의미:** 컨텍스트 예산·압축 동작을 한 파일로 남겨, 다음 Session 이나 회귀 점검의 입력으로 씁니다.

In [9]:
from datetime import datetime, timezone

report = {
"generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
"constants": {
"max": exaone.context_management.CONTEXT_LENGTH_MAX_TOKENS,
"recommended": exaone.context_management.CONTEXT_LENGTH_RECOMMENDED_TOKENS,
"tool_verbatim_cap": exaone.context_management.constants.CONTEXT_TOOL_VERBATIM_MAX_TOKENS,
},
"token_curve": token_curve,
"compaction": {
"before_tokens": token_curve[-1]["tokens_total"],
"after_compress_tokens": compressed_tokens,
"mock_summary_used": True,
},
}
path = out_dir / "compaction_report.json"
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())


saved: <cookbook-root>/recipes/track05_memory_and_long_context/_out/compaction_report.json


**출력 해석:** `saved:` 경로가 보이면 Session 1 의 상수·토큰 추이·압축 전후 수치가 파일로 저장된 것입니다.

## Session 2. Memory Ledger & Artifact

**테스트 시나리오** — `session_scenario.json` 에 적힌 **도구 3번을 순서대로 실행**하며 ledger·artifact·resume 을 봅니다(매 호출이 ledger 에 기록됨).

| 순서 | 도구 | 입력(args) | 도구의 역할 |
|---|---|---|---|
| 1 | `safe_lookup` | `topic="refund policy"` | 미니 정책 KB(refund/shipping/privacy)에서 주제를 찾아 본문 반환 |
| 2 | `safe_lookup` | `topic="shipping delay"` | 같은 도구로 다른 주제 조회 |
| 3 | `read_session_brief` | `max_entries=5` | 지금까지 ledger 에 쌓인 최근 hint 5건을 읽어옴 |

`exaone.memory` 는 2계층입니다 — **Ledger**(무슨 일이 있었는지 짧은 hint 의 시간순 목록) + **Artifact**(큰 결과의 원본). 큰 도구 결과는 artifact 로 빼고 ledger 엔 참조만 남겨, 컨텍스트엔 가벼운 요약만 포함합니다.

> **범위 주의:** Session 1 은 *대화 히스토리(메시지)* 자체를 줄이는 것이었고, Session 2 의 ledger/artifact 는 *도구 호출·결과 같은 "무슨 일이 있었나"* 를 기록합니다 — 그래서 `resume` 도 **ledger/artifact 상태(도구·세션 기록)를 되살리는 것**이지 대화 전체를 복원하는 게 아닙니다. (메커니즘은 범용이라 도구 결과가 아닌 다른 event 도 담을 수 있지만, Session 2 데모에선 도구 활동 위주로 구성되었습니다.)

### Session 2-1. `CORE_MEMORY_*` 설정

**하는 일:** 메모리 동작을 정하는 세 환경변수를 읽고, `default_memory_pair()` 가 그 값으로 ledger·artifact 를 만드는지 확인합니다.

**정상:** `CORE_MEMORY_ARTIFACT_MAX_ITEMS:`·`CORE_MEMORY_LEDGER_MAX_ENTRIES:`·`CORE_MEMORY_TOOL_RESULT_MIN_BYTES:` 가 출력됩니다.

**의미:** 2계층 메모리의 한도·임계값 설정입니다 — artifact 보관 개수, ledger 엔트리 수, 그리고 "도구 결과를 artifact 로 뺄지" 가르는 바이트 임계값. (1-1 의 컨텍스트 상수처럼 `.env` 의 `CORE_MEMORY_*` 에서 오고, 미설정이면 기본값)

In [10]:
import exaone.memory

artifacts0, ledger0 = exaone.memory.default_memory_pair()
print("CORE_MEMORY_ARTIFACT_MAX_ITEMS:", exaone.config.get_memory_artifact_max_items(), "->", artifacts0.max_items)
print("CORE_MEMORY_LEDGER_MAX_ENTRIES:", exaone.config.get_memory_ledger_max_entries(), "->", ledger0.max_entries)
print("CORE_MEMORY_TOOL_RESULT_MIN_BYTES:", exaone.config.get_memory_tool_result_min_bytes())


CORE_MEMORY_ARTIFACT_MAX_ITEMS: 256 -> 256
CORE_MEMORY_LEDGER_MAX_ENTRIES: 10000 -> 10000
CORE_MEMORY_TOOL_RESULT_MIN_BYTES: 2048


**출력 해석:** 세 값이 2계층 메모리의 한도를 정합니다.

- **ARTIFACT_MAX_ITEMS (256)** — artifact 저장소가 보관하는 최대 개수(초과 시 오래된 것부터 제거).
- **LEDGER_MAX_ENTRIES (10000)** — ledger 가 유지하는 최대 hint 수(초과 시 오래된 것부터 제거).
- **TOOL_RESULT_MIN_BYTES (2048)** — 도구 결과가 이 바이트 이상이면 **artifact 로 분리**하는 임계값(미만은 inline 유지). 2-2 의 `store_large_tool_result` 가 이 기준을 씁니다.

`->` 뒤 숫자는 `default_memory_pair()` 로 만든 객체에 그 설정이 그대로 반영됐다는 확인입니다.

### Session 2-2. Ledger + Artifact 기본 동작

**하는 일:** 두 가지를 직접 확인합니다 — ① ledger 가 최대 개수를 넘으면 오래된 hint 를 버리는지, ② `store_large_tool_result` 가 결과 크기에 따라 inline / artifact 로 갈라 저장하는지. 데모라 한도를 **일부러 작게**(`max_entries=5`, `max_items=3` — 기본값은 2-1 에서 10000, 256) 잡아, 적은 입력으로도 동작이 눈에 보이게 합니다.

**정상:** `ledger len (max 5):`·`small -> artifact:`·`large -> artifact:`·`artifact roundtrip keys:` 가 출력됩니다.

**의미:** `store_large_tool_result` 는 결과가 임계값(2048바이트(2KB)) **이상일 때만** 본문을 artifact 로 빼고 ledger 에 참조(`artifact:<id>`)를 남깁니다. 미만이면 `None` 을 반환하고 아무것도 저장하지 않습니다(작은 결과는 호출부에서 그대로 inline 으로 다룸).

In [11]:
demo_ledger = exaone.memory.InMemoryLedger(max_entries=5)
demo_art = exaone.memory.InMemoryArtifactStore(max_items=3)

# (en) Ledger keeps only the last max_entries hints (ring buffer).
# (kr) ledger 는 마지막 max_entries 개의 hint 만 유지한다 (링 버퍼).
for i in range(6):
    demo_ledger.append(event_type="turn", hint=f"turn-{i}", meta={"i": i})

small_id = exaone.memory.store_large_tool_result(result={"ok": True}, artifacts=demo_art, ledger=demo_ledger, tool_name="small_tool")
large_payload = {"rows": ["x" * 300 for _ in range(20)]}
large_id = exaone.memory.store_large_tool_result(result=large_payload, artifacts=demo_art, ledger=demo_ledger, tool_name="analytics_export")
print("ledger len (max 5):", len(demo_ledger))
print("small -> artifact:", small_id, "(None = stayed inline)")
print("large -> artifact:", large_id)
if large_id:
    got = demo_art.get(large_id)
    # (kr) 저장 전 large_payload 와 값이 같은지 비교 (dict == dict, 값 기준)
    print("artifact roundtrip == original:", got.payload == large_payload if got else None)


ledger len (max 5): 5
small -> artifact: None (None = stayed inline)
large -> artifact: 23ce381a-be31-4a00-9752-e3b9dd8f811d
artifact roundtrip == original: True


**출력 해석:**
- **ledger len (max 5): 5** — turn 을 6개 넣었지만 최대 5개라, 가장 오래된 것이 밀려나고 5개만 남습니다.
- **small -> artifact: None** — `{"ok":True}` 는 2048바이트(2KB) 미만이라 artifact 로 빼지 않습니다(저장 안 함, inline 유지).
- **large -> artifact: <id>** — 큰 결과는 artifact 로 저장되고 id 가 반환됩니다. 이때만 ledger 에도 참조 hint 가 추가됩니다.
- **artifact roundtrip == original: True** — 그 id 로 `get(id).payload` 한 값이 저장 전 `large_payload` 와 같습니다 — 원본이 온전히 복원됩니다.

### Session 2-3. 세션 스냅샷 save / load

**하는 일:** ledger·artifact 를 JSON 으로 직렬화/복원하는 `snapshot_to_dict`·`restore_from_dict` 를 정의하고, `_out/session_state.json` 이 있으면 거기서 **복원(resume)**, 없으면 `session_start` 로 **새 세션**을 시작합니다.

**정상:** 첫 실행은 `new session:`, 저장 파일이 있으면 `resumed session: … | ledger entries: N` 이 출력됩니다.

**의미:** resume 의 핵심 — 메모리를 파일로 떨궜다가(2-6) 다음 실행에 그대로 되살립니다. `restore_from_dict` 는 원래 `id` 까지 보존해 복원하므로 ledger 의 참조(`artifact:<id>`)도 그대로 이어집니다.

In [12]:
scenario = json.loads((DATA / "session_scenario.json").read_text(encoding="utf-8"))
print("시나리오: ledger 도구", len(scenario.get("tool_calls", [])), "건 (session_scenario.json)")
for step in scenario.get("tool_calls", []):
    print(f"  {step['tool']}:", step.get("args"))
SESSION_ID = scenario["session_id"]
STATE_PATH = out_dir / "session_state.json"


def snapshot_to_dict(ledger, artifacts):
    return {
        "session_id": SESSION_ID,
        "ledger": [{"id": e.id, "event_type": e.event_type, "hint": e.hint, "artifact_id": e.artifact_id, "meta": dict(e.meta)} for e in ledger.as_list()],
        "artifacts": [{"id": a.id, "hint": a.hint, "payload": a.payload, "meta": dict(a.meta)} for aid in artifacts.snapshot_ids_ordered() if (a := artifacts.get(aid)) is not None],
    }


def restore_from_dict(data):
    art = exaone.memory.InMemoryArtifactStore(max_items=exaone.config.get_memory_artifact_max_items())
    led = exaone.memory.InMemoryLedger(max_entries=exaone.config.get_memory_ledger_max_entries())
    for row in data.get("artifacts", []):
        art.put(row["payload"], hint=row.get("hint", ""), meta=row.get("meta"), artifact_id=row["id"])
    for row in data.get("ledger", []):
        led.append(event_type=row["event_type"], hint=row["hint"], artifact_id=row.get("artifact_id"), meta=row.get("meta"), entry_id=row["id"])
    return art, led


if STATE_PATH.is_file():
    saved = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    artifacts, ledger = restore_from_dict(saved)
    resumed = True
    print("resumed session:", SESSION_ID, "| ledger entries:", len(ledger))
else:
    artifacts = exaone.memory.InMemoryArtifactStore(max_items=exaone.config.get_memory_artifact_max_items())
    ledger = exaone.memory.InMemoryLedger(max_entries=exaone.config.get_memory_ledger_max_entries())
    ledger.append(event_type="session_start", hint=f"id={SESSION_ID}", meta={})
    resumed = False
    print("new session:", SESSION_ID)


시나리오: ledger 도구 3 건 (session_scenario.json)
  safe_lookup: {'topic': 'refund policy'}
  safe_lookup: {'topic': 'shipping delay'}
  read_session_brief: {'max_entries': 5}
new session: track05-demo


**출력 해석:** 처음 돌리면 저장 파일이 없어 `new session: track05-demo` (resumed=False) — `session_start` hint 1개로 시작합니다. 2-6 에서 `session_state.json` 을 저장한 뒤 노트북을 다시 실행하면 이 셀이 `resumed session:` 으로 바뀌고 ledger entries 가 누적됩니다.

### Session 2-4. 시나리오 실행 — 도구 호출을 ledger 에 기록

**하는 일:** 2-3 에서 만든 ledger 에, 시나리오의 도구 3건(`safe_lookup`×2, `read_session_brief`)을 실제 `ToolRegistry` 로 실행합니다. 매 호출 결과를 `_log_tool_result` 로 ledger 에 남깁니다(LLM 불필요).

**입력:** `data/session_scenario.json`

**정상:** 도구별 `-> ok=…`, 그리고 `ledger entries after tools:` 가 출력됩니다.

**의미:** 모든 도구 결과가 ledger 에 hint(성공/빈결과/실패)로, 큰 결과면 artifact 로도 기록됩니다. `read_session_brief` 는 그 ledger 를 거꾸로 읽어오는 도구 — 에이전트가 "지금까지 뭐 했지?"를 조회하는 셈입니다.

In [13]:
from typing import Any

from exaone.config import get_memory_tool_result_min_bytes
from exaone.memory import store_large_tool_result
from exaone.tools import ToolRegistry, tool_from_callable
from exaone.tools.tool_result import ToolResult

_MOCK_KB = {
"refund": "Refunds are allowed within 14 days with receipt. " + "(policy detail) " * 200,  # (kr) 일부러 크게 -> 큰 결과가 artifact 로 빠지는 경우 시연
"shipping": "Standard shipping takes 3-5 business days.",
"privacy": "We do not sell personal data to third parties.",
}


def _ledger_brief_lines(ledger, max_entries):
    return [f"- [{e.event_type}] {e.hint}" for e in ledger.as_list()[-max_entries:]]


# (en) Persist every tool result to ledger (hint) + artifact (payload if large).
# (kr) 모든 도구 결과를 ledger(hint)와 artifact(큰 payload)에 남긴다.
def _log_tool_result(*, ledger, artifacts, tool_name, result):
    store_large_tool_result(
        result=result, artifacts=artifacts, ledger=ledger,
        event_type="tool_result", tool_name=tool_name, min_bytes=get_memory_tool_result_min_bytes(),
    )
    outcome = result.get("outcome")
    if outcome == "empty":
        suffix = " empty"
    elif outcome == "success" or result.get("ok"):
        suffix = " ok"
    else:
        suffix = f" fail:{(result.get('error') or '')[:80]}"
    ledger.append(event_type="tool", hint=((result.get("source") or tool_name) + suffix)[:500], meta={"tool": tool_name})


def build_tool_registry(*, ledger, artifacts):
    reg = ToolRegistry()

    def exec_read_brief(_name, args):
        n = int(args.get("max_entries") or 10)
        lines = _ledger_brief_lines(ledger, max(1, min(n, 50)))
        out = ToolResult.success(content="\n".join(lines) if lines else "(empty ledger)",
                                 source="read_session_brief", metadata={"entry_count": len(lines)}).to_dict()
        _log_tool_result(ledger=ledger, artifacts=artifacts, tool_name="read_session_brief", result=out)
        return out

    reg.register(tool_from_callable("read_session_brief", {
        "type": "function", "function": {
            "name": "read_session_brief",
            "description": "Read recent ledger hints (tool calls, session events).",
            "parameters": {"type": "object", "properties": {"max_entries": {"type": "integer", "default": 10}}}}},
        exec_read_brief))

    def exec_safe_lookup(_name, args):
        topic = (args.get("topic") or "").strip().lower()
        if not topic:
            out = ToolResult.validation_error(source="safe_lookup", error="topic required").to_dict()
            _log_tool_result(ledger=ledger, artifacts=artifacts, tool_name="safe_lookup", result=out)
            return out
        for key, text in _MOCK_KB.items():
            if key in topic:
                out = ToolResult.success(content=text, source="safe_lookup", metadata={"matched": key}).to_dict()
                _log_tool_result(ledger=ledger, artifacts=artifacts, tool_name="safe_lookup", result=out)
                return out
        out = ToolResult.empty(source="safe_lookup", reason="no matching topic; try refund, shipping, or privacy").to_dict()
        _log_tool_result(ledger=ledger, artifacts=artifacts, tool_name="safe_lookup", result=out)
        return out

    reg.register(tool_from_callable("safe_lookup", {
        "type": "function", "function": {
            "name": "safe_lookup",
            "description": "Mock policy lookup; each call is logged to ledger/artifact.",
            "parameters": {"type": "object", "properties": {"topic": {"type": "string"}}, "required": ["topic"]}}},
        exec_safe_lookup))
    return reg


registry = build_tool_registry(ledger=ledger, artifacts=artifacts)
run_log = []
for step in scenario["tool_calls"]:
    out = registry.execute(step["tool"], step["args"])
    run_log.append({"tool": step["tool"], "args": step["args"], "ok": out.get("ok"), "source": out.get("source")})
    print(step["tool"], "-> ok=", out.get("ok"), "source=", out.get("source"))
print("ledger entries after tools:", len(ledger))
print("artifacts created:", len(artifacts.snapshot_ids_ordered()))


safe_lookup -> ok= True source= safe_lookup
safe_lookup -> ok= True source= safe_lookup
read_session_brief -> ok= True source= read_session_brief
ledger entries after tools: 5
artifacts created: 1


**출력 해석:** 도구 3건이 각각 `ok=True` 로 실행되고, 결과가 ledger·artifact 에 기록됩니다. (절대 합계는 resume·재실행으로 누적돼 **실행마다 달라지니**, 호출당 기여분으로 아래 설명)

- **ledger**    
    - 2-3 의 `session_start` 가 1개
    - refund 결과는 커서(>2048바이트) `tool_result`(artifact 참조) + `tool`로 **2개**
    - shipping·brief 는 작아서 `tool` 이 각각 **1개**씩. 
    - 첫 실행이면 1+2+1+1 = 5, 다시 돌리면 +4 씩 누적
- **artifact** 
    - 큰 refund 결과만 **artifact 로 분리**되고, 나머지 작은 결과(shipping·brief)는 inline 유지
    - 그 참조는 2-5 ledger 에 `[tool_result] artifact:<id>` 로 보입니다. 
    - 첫 실행 1개, 재실행마다 새 artifact 추가

### Session 2-5. Ledger brief 확인

**하는 일:** ledger 의 hint 를 그대로 출력해, 지금까지 무슨 일이 있었는지 한눈에 봅니다.

**정상:** `recent ledger:` 아래 `[event_type] hint` 줄들이 출력됩니다.

**의미:** ledger 는 "무슨 일이 있었나"를 짧은 hint 로 시간순 기록한 것 — 이걸 그대로 컨텍스트에 싸게 넣어 에이전트가 자기 이력을 참조합니다(2-4 `read_session_brief` 가 하던 일).

In [14]:
print("recent ledger:")
entries = ledger.as_list()
# (kr) 10개 이하면 전부, 넘으면 처음 5 + 생략 + 마지막 5
rows = entries if len(entries) <= 10 else entries[:5] + [None] + entries[-5:]
for e in rows:
    if e is None:
        print(f"  ... ({len(entries) - 10} more)")
    else:
        print(f"  [{e.event_type}] {e.hint}")


recent ledger:
  [session_start] id=track05-demo
  [tool_result] artifact:48b9fd24-6e23-44d1-8399-537cd52d18a5
  [tool] safe_lookup ok
  [tool] safe_lookup ok
  [tool] read_session_brief ok


**출력 해석:** `[session_start]` 뒤로 2-4 의 도구 기록이 보입니다 — 큰 refund 는 `[tool_result] artifact:<id>`(원본은 artifact, ledger 엔 참조만), 나머지는 `[tool] safe_lookup ok` 같은 결과 요약 hint. ledger 만 봐도 흐름이 읽히고, 큰 내용은 필요할 때 artifact id 로 꺼냅니다.

### Session 2-6. 산출물 — `session_state.json` (재실행 시 resume)

**하는 일:** 현재 ledger·artifact·run_log 를 `session_state.json` 으로 저장합니다(2-3 의 `snapshot_to_dict` 사용).

**정상:** `saved:` 경로와 `entry_count: … | artifact_count: …` 가 출력됩니다.

**의미:** 이 파일이 있으면 다음 실행에서 2-3 이 그대로 복원(resume)합니다 — 메모리를 세션 간에 이어가는 마무리 단계입니다.

In [15]:
payload = snapshot_to_dict(ledger, artifacts)
payload["generated_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
payload["resumed_at_start"] = resumed
payload["run_log"] = run_log
STATE_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", STATE_PATH.resolve())
print("entry_count:", len(payload["ledger"]), "| artifact_count:", len(payload["artifacts"]))
print("tip: 노트북을 다시 실행하면 위 save/load 셀이 resumed session 으로 더 많은 entry 를 출력합니다.")


saved: <cookbook-root>/recipes/track05_memory_and_long_context/_out/session_state.json
entry_count: 5 | artifact_count: 1
tip: 노트북을 다시 실행하면 위 save/load 셀이 resumed session 으로 더 많은 entry 를 출력합니다.


**출력 해석:** `saved:` 경로가 보이면 메모리 상태가 파일로 저장된 것입니다. `entry_count`·`artifact_count` 는 첫 실행이면 5·1(2-4 기준), 노트북을 다시 실행하면 2-3 이 이 파일에서 resume 해 `resumed_at_start: true` + 값이 누적됩니다.

## 체크포인트

- [ ] Session 1 `compaction_report.json` — `before_tokens` > `after_compress_tokens`, 압축 후 토큰이 줄어든다.
- [ ] Session 1 도구 verbatim cap — 큰 도구 결과가 `needs verbatim cap: True` 로 잡힌다.
- [ ] Session 2 첫 실행: `resumed_at_start: false`, `entry_count >= 4`.
- [ ] Session 2 재실행: `resumed_at_start: true`, entry_count 증가 (세션 복원).

**다음:** Track 06 — Orchestration & Multi-Agent
